# QuantumEdge — Exploratory Data Analysis

**BTCUSDT 5-minute OHLCV** — 2017-09-01 to 2025-12-31

Covers: data overview, price action, volume analysis, volatility, missing data, regime detection.

In [ ]:
import sys
from pathlib import Path
# Add project root to path (works regardless of execution dir)
root = Path.cwd()
while not (root / 'research').exists():
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

from research.data.loader import load_parquet, OHLCV_COLUMNS

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

In [ ]:
df = load_parquet()
print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0]} to {df.index[-1]}')
print(f'Total years: {(df.index[-1] - df.index[0]).days / 365.25:.2f}')
print(f'Columns: {list(df.columns)}')
df.head()

## 1. Data Quality & Missing Data

In [ ]:
# Missing data summary
missing = df.isna().sum()
missing_pct = 100 * missing / len(df)
print('Missing values per column:')
for col in df.columns:
    print(f'  {col:>8}: {missing[col]:>6} ({missing_pct[col]:.3f}%)')

# Missing by year
missing_by_year = df.isna().groupby(df.index.year).sum()
print('\nMissing by year:')
display(missing_by_year)

In [ ]:
# Gap analysis — find gaps > 5 min
time_diffs = df.index.to_series().diff().dt.total_seconds()
gaps = time_diffs[time_diffs > 300]
print(f'Total gaps (>5 min): {len(gaps)}')
print(f'Largest gap: {gaps.max() / 3600:.1f} hours')
print(f'Gap distribution (minutes):')
print(gaps.describe().to_string())

if len(gaps) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].hist(gaps / 60, bins=50, edgecolor='black')
    axes[0].set_xlabel('Gap duration (minutes)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Gap Duration Distribution')
    
    # Gaps by year
    gap_years = gaps.index.year.value_counts().sort_index()
    axes[1].bar(gap_years.index, gap_years.values)
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Number of gaps')
    axes[1].set_title('Gaps by Year')
    plt.tight_layout()
    plt.show()

## 2. Price Action

In [ ]:
# Close price over time
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['close'], linewidth=0.5, color='#1f77b4')
ax.set_title('BTCUSDT Close Price (5m candles)', fontsize=14)
ax.set_ylabel('Price (USDT)')
ax.set_xlabel('Date')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Daily returns
daily_close = df['close'].resample('1D').last()
daily_returns = daily_close.pct_change().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(daily_close.index, daily_close, linewidth=0.8)
axes[0].set_title('Daily Close Price')
axes[0].set_ylabel('Price (USDT)')

axes[1].plot(daily_returns.index, daily_returns, linewidth=0.5, alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--', linewidth=0.5)
axes[1].set_title('Daily Returns')
axes[1].set_ylabel('Return')

plt.tight_layout()
plt.show()

print(f'Daily return stats:')
print(f'  Mean: {daily_returns.mean():.4f}')
print(f'  Std:  {daily_returns.std():.4f}')
print(f'  Min:  {daily_returns.min():.4f}')
print(f'  Max:  {daily_returns.max():.4f}')
print(f'  Skew: {daily_returns.skew():.4f}')
print(f'  Kurt: {daily_returns.kurtosis():.4f}')

In [ ]:
# Returns distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(daily_returns, bins=200, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Daily Returns Distribution')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')

# QQ plot against normal
from scipy import stats
stats.probplot(daily_returns, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (vs Normal)')

plt.tight_layout()
plt.show()

# Normality test
stat, p = stats.normaltest(daily_returns.dropna())
print(f'D\'Agostino-Pearson normality test: stat={stat:.2f}, p={p:.2e}')
print(f'  → Returns are {"NOT " if p < 0.05 else ""}normally distributed (α=0.05)')

## 3. Volume Analysis

In [ ]:
# Volume statistics
print('Volume statistics (BTC per 5m candle):')
print(df['volume'].describe().to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Volume over time
axes[0].plot(df.index, df['volume'], linewidth=0.3, alpha=0.5, color='green')
axes[0].set_title('Volume Over Time')
axes[0].set_ylabel('Volume (BTC)')
axes[0].set_yscale('log')

# Volume distribution
axes[1].hist(df['volume'], bins=200, edgecolor='black', alpha=0.7, color='green')
axes[1].set_title('Volume Distribution')
axes[1].set_xlabel('Volume (BTC)')
axes[1].set_ylabel('Frequency')
axes[1].set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Volume by hour of day (UTC)
df['hour'] = df.index.hour
hourly_vol = df.groupby('hour')['volume'].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(hourly_vol.index, hourly_vol.values, edgecolor='black')
ax.set_title('Average Volume by Hour of Day (UTC)')
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('Avg Volume (BTC)')
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

# Volume by day of week
df['dow'] = df.index.dayofweek
dow_vol = df.groupby('dow')['volume'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
print('Average volume by day of week:')
for i, v in enumerate(dow_vol):
    print(f'  {days[i]}: {v:.1f} BTC')

In [ ]:
# Volume-price correlation
df['ret_5m'] = df['close'].pct_change()
df['vol_abs_ret'] = df['volume'] * df['ret_5m'].abs()

corr = df[['close', 'volume', 'ret_5m']].corr()
print('Correlation matrix:')
display(corr)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Price-Volume Correlation')
plt.tight_layout()
plt.show()

## 4. Volatility Analysis

In [ ]:
# ATR (Average True Range) — 14 period
high, low, close = df['high'], df['low'], df['close']
tr = pd.concat([
    high - low,
    (high - close.shift()).abs(),
    (low - close.shift()).abs()
], axis=1).max(axis=1)
atr_14 = tr.rolling(14).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True)

axes[0].plot(df.index, df['close'], linewidth=0.5)
axes[0].set_title('BTCUSDT Close Price')
axes[0].set_ylabel('Price (USDT)')

axes[1].plot(df.index, atr_14, linewidth=0.5, color='orange')
axes[1].set_title('ATR(14) — Average True Range')
axes[1].set_ylabel('ATR (USDT)')

plt.tight_layout()
plt.show()

In [ ]:
# Volatility regimes — classify by ATR percentile
atr_pct = atr_14.rank(pct=True)
df['regime'] = pd.cut(atr_pct, bins=[0, 0.25, 0.75, 1.0],
                       labels=['Low Vol', 'Normal', 'High Vol'])

regime_counts = df['regime'].value_counts()
print('Volatility regime distribution:')
for regime in ['Low Vol', 'Normal', 'High Vol']:
    pct = 100 * regime_counts[regime] / len(df)
    print(f'  {regime}: {pct:.1f}%')

# Plot regimes on price
fig, ax = plt.subplots(figsize=(16, 5))
colors = {'Low Vol': 'green', 'Normal': 'gray', 'High Vol': 'red'}
for regime, color in colors.items():
    mask = df['regime'] == regime
    ax.scatter(df.index[mask], df['close'][mask], 
               c=color, s=0.5, alpha=0.3, label=regime)
ax.set_title('BTCUSDT with Volatility Regimes')
ax.set_ylabel('Price (USDT)')
ax.set_yscale('log')
ax.legend(markerscale=10)
plt.tight_layout()
plt.show()

In [ ]:
# Rolling volatility (30-day annualized)
daily_vol = daily_returns.rolling(30).std() * np.sqrt(365)

fig, ax = plt.subplots(figsize=(16, 4))
ax.fill_between(daily_vol.index, daily_vol, alpha=0.3)
ax.plot(daily_vol.index, daily_vol, linewidth=0.8)
ax.axhline(daily_vol.mean(), color='red', linestyle='--', label=f'Mean: {daily_vol.mean():.1%}')
ax.set_title('30-Day Rolling Annualized Volatility')
ax.set_ylabel('Volatility')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Annualized volatility stats:')
print(f'  Mean: {daily_vol.mean():.1%}')
print(f'  Median: {daily_vol.median():.1%}')
print(f'  Min: {daily_vol.min():.1%}')
print(f'  Max: {daily_vol.max():.1%}')

## 5. Market Regime Detection (Simple)

In [ ]:
# Simple bull/bear/sideways classification using SMA200
sma_50 = daily_close.rolling(50).mean()
sma_200 = daily_close.rolling(200).mean()

daily = pd.DataFrame({'close': daily_close, 'sma_50': sma_50, 'sma_200': sma_200})
daily['regime'] = 'Sideways'
daily.loc[daily['close'] > daily['sma_200'] * 1.05, 'regime'] = 'Bull'
daily.loc[daily['close'] < daily['sma_200'] * 0.95, 'regime'] = 'Bear'

regime_pct = daily['regime'].value_counts(normalize=True) * 100
print('Market regime distribution (daily):')
for regime in ['Bull', 'Bear', 'Sideways']:
    print(f'  {regime}: {regime_pct.get(regime, 0):.1f}%')

fig, ax = plt.subplots(figsize=(16, 5))
colors = {'Bull': 'green', 'Bear': 'red', 'Sideways': 'gray'}
for regime, color in colors.items():
    mask = daily['regime'] == regime
    ax.scatter(daily.index[mask], daily['close'][mask], 
               c=color, s=3, alpha=0.5, label=regime)
ax.plot(daily.index, daily['sma_200'], color='blue', linewidth=1, alpha=0.7, label='SMA 200')
ax.set_title('Market Regimes (Bull/Bear/Sideways)')
ax.set_ylabel('Price (USDT)')
ax.set_yscale('log')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Key Statistics Summary

In [ ]:
print('=' * 60)
print('QUANTUMEDGE — BTCUSDT 5m DATA SUMMARY')
print('=' * 60)
print(f'\nData period:     {df.index[0]} to {df.index[-1]}')
print(f'Total candles:   {len(df):,}')
print(f'Trading days:    {len(daily_close):,}')
print(f'Missing candles: {df.isna().any(axis=1).sum():,} ({100*df.isna().any(axis=1).sum()/len(df):.2f}%)')
print(f'\nPrice range:     ${df["close"].min():.2f} — ${df["close"].max():.2f}')
print(f'Avg daily return: {daily_returns.mean()*100:.3f}%')
print(f'Daily vol (σ):    {daily_returns.std()*100:.2f}%')
print(f'Ann. volatility:  {daily_returns.std()*np.sqrt(365)*100:.1f}%')
print(f'Sharpe (daily):   {daily_returns.mean()/daily_returns.std()*np.sqrt(365):.2f}')
print(f'\nVolume range:    {df["volume"].min():.1f} — {df["volume"].max():.1f} BTC')
print(f'Avg volume/candle: {df["volume"].mean():.1f} BTC')
print(f'\nRegime split:')
for regime in ['Bull', 'Bear', 'Sideways']:
    pct = regime_pct.get(regime, 0)
    print(f'  {regime:>10}: {pct:.1f}%')
print('=' * 60)